# Lab 03 -- What RAG Is Made Of

Week 5 (content: Context Engineering, week 4) · Narxoz University

A retrieval-augmented generation system is four small, separately-testable pieces: **chunking** (cut documents into retrievable units), **embedding** (turn text into vectors), **retrieval** (find the vectors closest to a question), and **generation** (answer using only what retrieval found). This lab builds all four by hand, on four short policy articles from a fictional bank -- same domain as Lab 01's support queue.

**No API key. No installation. No cost.** Two small open models are downloaded inside this notebook: an embedding model for retrieval, and an instruction-following model for generation.

## Setup

Open this notebook in Google Colab and run the next cell first -- it installs the one package Colab does not ship with. Runtime → Run all is fine for this cell only; from Part 1 onward, run cells one at a time so you can write your prediction before you see the answer.

In [1]:
!pip install -q sentence-transformers

In [2]:
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from transformers import AutoModelForCausalLM, AutoTokenizer

I ran this notebook locally in Jupyter on my laptop (Windows 11, RTX 3050 Laptop 4 GB), not in Colab. I used a virtual environment (`.venv`), the package versions are in `requirements.txt`. The lab code is not changed, so generation runs on CPU like in the original.

In [3]:
import platform, sys, transformers, sentence_transformers

print(f"python {sys.version.split()[0]} on {platform.system()} {platform.release()}")
print(f"torch {torch.__version__}, transformers {transformers.__version__}, sentence-transformers {sentence_transformers.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

python 3.12.0 on Windows 11
torch 2.5.1+cu121, transformers 5.12.1, sentence-transformers 6.1.0
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## Part 0 -- the corpus

Four articles from the bank's policy pages, all about payment cards -- deliberately near-neighbours, so a retrieval system has to tell them apart, not just notice the word "card". Each article is a list of sentences; `needle_idx` marks the one sentence that actually answers that article's two questions (`literal`, a direct restatement, and `paraphrase`, the same need in different words).

Read all four before running the next cell.

In [4]:
CORPUS = [
    {
        "id": "cards-lost",
        "needle_idx": 5,
        "sentences": [
            "If your card is lost or stolen, block it at once in the mobile app under Cards, then Block, or call the round-the-clock hotline printed on the back of the card.",
            "Blocking takes effect within 60 seconds, after which the card cannot be used for payments or cash withdrawals.",
            "You are not liable for any transaction made after you report the loss.",
            "Earlier transactions are reviewed one by one, and the Bank refunds those you did not authorise, provided you never shared your PIN.",
            "A replacement card is issued free of charge and is ready at your chosen branch within 5 working days.",
            "If you need a card the same day, the branch can issue an instant card for 3000 tenge.",
        ],
        "queries": {
            "literal": "How much does an instant replacement card cost at the branch?",
            "paraphrase": "I lost my wallet this morning and need a working card by tonight - what will it cost me?",
        },
    },
    {
        "id": "cards-pin-block",
        "needle_idx": 3,
        "sentences": [
            "After three wrong PIN entries in a row, the card is blocked automatically to protect your money.",
            "A card blocked this way is not lost: your funds and your account are untouched.",
            "To unblock it, open the app, choose the card, and confirm with the six-digit code we send by SMS.",
            "Unblocking in the app is free and takes effect within 2 minutes.",
            "If you have no access to the app, any branch can unblock the card on presentation of your ID, also free of charge.",
            "After unblocking, the wrong-attempt counter resets to zero, and you have three new attempts.",
        ],
        "queries": {
            "literal": "How long does it take for the card to work again after I unblock it in the app?",
            "paraphrase": "I mistyped my code at the till and the card stopped working - once I sort it out on my phone, how soon can I pay with it again?",
        },
    },
    {
        "id": "cards-expiry",
        "needle_idx": 4,
        "sentences": [
            "Your card is valid until the last day of the month printed on its face.",
            "We start reissuing it automatically 30 days before that date, so you do not need to apply.",
            "The new card keeps the same number and PIN, but has a new expiry date and a new security code.",
            "It is delivered to your home branch, and we notify you by SMS when it is ready.",
            "Uncollected cards are destroyed after 60 days for security reasons, and a new one then costs 1500 tenge.",
        ],
        "queries": {
            "literal": "How many days do I have to collect a reissued card before it is destroyed?",
            "paraphrase": "My new card has been waiting at the branch for ages and I keep forgetting to go - is there a point where they throw it away?",
        },
    },
    {
        "id": "cards-contactless",
        "needle_idx": 0,
        "sentences": [
            "Contactless payments up to 15000 tenge do not require a PIN.",
            "After five contactless payments in a row without a PIN, the terminal asks for the PIN once, and the counter starts again.",
            "You can lower the limit, or switch contactless off completely, in the app under Card settings.",
            "Changes to the limit take effect immediately and are free.",
            "Payments above the limit always require a PIN, whether or not you have changed the setting.",
        ],
        "queries": {
            "literal": "What is the maximum contactless payment that does not need a PIN?",
            "paraphrase": "Up to what sum can I simply hold my card near the reader and walk away without typing anything?",
        },
    },
]

for article in CORPUS:
    print(f"{article['id']}: {len(article['sentences'])} sentences, needle is #{article['needle_idx']}")

cards-lost: 6 sentences, needle is #5
cards-pin-block: 6 sentences, needle is #3
cards-expiry: 5 sentences, needle is #4
cards-contactless: 5 sentences, needle is #0


## Part 1 -- chunking

A *chunk* is the unit retrieval works on. Two ways to cut these four articles into chunks:

- **article-level**: one chunk per article (its sentences joined into one block).
- **sentence-level**: one chunk per sentence.

✏️ **Predict, before running the next cell**: from the sentence counts printed in Part 0, how many chunks will each strategy produce in total?

**My prediction (Part 1):** 4 article-level chunks (one per article) and 22 sentence-level chunks (6 + 6 + 5 + 5).

In [5]:
def chunk_by_article(corpus):
    return [
        {"id": a["id"], "article_id": a["id"], "text": " ".join(a["sentences"])}
        for a in corpus
    ]


def chunk_by_sentence(corpus):
    return [
        {"id": f"{a['id']}:{i}", "article_id": a["id"], "text": s}
        for a in corpus
        for i, s in enumerate(a["sentences"])
    ]


article_chunks = chunk_by_article(CORPUS)
sentence_chunks = chunk_by_sentence(CORPUS)
print(f"article-level chunks: {len(article_chunks)}")
print(f"sentence-level chunks: {len(sentence_chunks)}")

article-level chunks: 4
sentence-level chunks: 22


**Result:** 4 and 22, same as my prediction.

## Part 2 -- embedding and retrieval

The embedding model turns text into a vector; cosine similarity measures how close two vectors point. `multilingual-e5-small` (the model below) was trained with a convention: prefix every *document* with `"passage: "` and every *question* with `"query: "` -- without the prefixes, retrieval quality drops noticeably. This is a property of this specific model family, not a universal rule.

✏️ **Predict, before running**: for the `cards-lost` article's `literal` query (`"How much does an instant replacement card cost at the branch?"`), which of the four articles do you expect **article-level** retrieval to return? And which exact **sentence** (by its content, not its number) do you expect **sentence-level** retrieval to return? Read the four articles in Part 0 to answer -- don't guess blind.

**My prediction (Part 2, literal):** article-level will return cards-lost. Sentence-level will return "If you need a card the same day, the branch can issue an instant card for 3000 tenge." because the query has the same words: instant, card, branch, cost.

In [6]:
embedder = SentenceTransformer("intfloat/multilingual-e5-small")

article_vecs = embedder.encode(
    [f"passage: {c['text']}" for c in article_chunks], convert_to_tensor=True
)
sentence_vecs = embedder.encode(
    [f"passage: {c['text']}" for c in sentence_chunks], convert_to_tensor=True
)


def retrieve(chunks, chunk_vecs, query, k=3):
    q_vec = embedder.encode(f"query: {query}", convert_to_tensor=True)
    scores = cos_sim(q_vec, chunk_vecs)[0]
    top = torch.topk(scores, k=min(k, len(chunks)))
    return [(chunks[i], float(top.values[j])) for j, i in enumerate(top.indices)]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
demo_article = CORPUS[0]  # cards-lost
query = demo_article["queries"]["literal"]
print(f"query: {query}\n")

top_article = retrieve(article_chunks, article_vecs, query, k=1)[0]
print(f"article-level retrieved: {top_article[0]['article_id']} (score {top_article[1]:.3f})\n")

top_sentences = retrieve(sentence_chunks, sentence_vecs, query, k=3)
for chunk, score in top_sentences:
    print(f"sentence-level [{score:.3f}] {chunk['id']}: {chunk['text']}")

query: How much does an instant replacement card cost at the branch?

article-level retrieved: cards-lost (score 0.869)

sentence-level [0.851] cards-lost:4: A replacement card is issued free of charge and is ready at your chosen branch within 5 working days.
sentence-level [0.848] cards-lost:5: If you need a card the same day, the branch can issue an instant card for 3000 tenge.
sentence-level [0.828] cards-expiry:4: Uncollected cards are destroyed after 60 days for security reasons, and a new one then costs 1500 tenge.


**Result:** article-level returned cards-lost (0.869), so this part was right.

Sentence-level was wrong. First place is cards-lost:4 "A replacement card is issued free of charge..." (0.851). The correct sentence about 3000 tenge is second with 0.848, so the difference is only 0.003. I think the model matched "replacement card" and "branch" more than "instant". If we took only the top 1 result, the model would get the wrong fact (free of charge).

Now the harder version: the same article's **paraphrase** query -- the same need, described without reusing the article's own words.

✏️ **Predict again before running**: do you expect the paraphrase to change either answer above?

**My prediction (Part 2, paraphrase):** article-level will still be cards-lost, because the whole article is about a lost card. Sentence-level can change, because the paraphrase does not have the words instant and branch.

In [8]:
query = demo_article["queries"]["paraphrase"]
print(f"query: {query}\n")

top_article = retrieve(article_chunks, article_vecs, query, k=1)[0]
print(f"article-level retrieved: {top_article[0]['article_id']} (score {top_article[1]:.3f})\n")

top_sentences = retrieve(sentence_chunks, sentence_vecs, query, k=3)
for chunk, score in top_sentences:
    print(f"sentence-level [{score:.3f}] {chunk['id']}: {chunk['text']}")

query: I lost my wallet this morning and need a working card by tonight - what will it cost me?

article-level retrieved: cards-expiry (score 0.881)

sentence-level [0.864] cards-expiry:4: Uncollected cards are destroyed after 60 days for security reasons, and a new one then costs 1500 tenge.
sentence-level [0.861] cards-lost:4: A replacement card is issued free of charge and is ready at your chosen branch within 5 working days.
sentence-level [0.860] cards-pin-block:1: A card blocked this way is not lost: your funds and your account are untouched.


**Result:** article-level returned cards-expiry (0.881). I did not expect this, I thought it would stay cards-lost.

Sentence-level: first is cards-expiry:4 "...a new one then costs 1500 tenge", then cards-lost:4, then cards-pin-block:1. The correct sentence is not in the top 3 at all. I guessed it could change, but it was worse than I thought. Without instant and branch the question is mostly "new card + how much", and the 1500 tenge sentence fits that better. The three scores are very close (0.864, 0.861, 0.860).

## Part 3 -- generation, without and with retrieval

`Qwen2.5-1.5B-Instruct` has never seen this fictional bank's policy -- it cannot know the real answer, only produce a fluent-sounding guess. Retrieval's job is to hand it the one sentence it needs so it does not have to guess.

✏️ **Predict, before running the next two cells**: will the no-context answer refuse to guess, or will it state a specific but made-up number?

**My prediction (Part 3):** the model will not refuse. It will say some specific price that it made up, because small models usually try to answer anyway.

In [9]:
GEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, dtype=torch.float32)
gen_model.eval()


def generate(prompt, max_new_tokens=90):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt")
    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [10]:
query = demo_article["queries"]["literal"]

no_context_answer = generate(f"Answer in one or two sentences: {query}")
print("--- WITHOUT retrieved context ---")
print(no_context_answer)

--- WITHOUT retrieved context ---
I'm sorry, but I cannot provide you with that information as it may violate our terms of service to answer questions about specific products or services. However, you can usually find this information on the official website of the company or contact their customer support for assistance.


**Result:** my prediction was wrong. The model refused to answer, but the reason was made up: it said it "may violate our terms of service" and told me to check the company website. It did not give 3000 tenge.

In [11]:
needle = demo_article["sentences"][demo_article["needle_idx"]]

with_context_prompt = (
    "Answer the question using ONLY the fact below. "
    "If the fact does not answer the question, say you don't know.\n\n"
    f"Fact: {needle}\n\nQuestion: {query}"
)
with_context_answer = generate(with_context_prompt)
print("--- WITH retrieved context ---")
print(with_context_answer)

--- WITH retrieved context ---
The cost of an instant replacement card at the branch is 3000 tenge.


**Result:** "The cost of an instant replacement card at the branch is 3000 tenge." This is correct and it comes from the retrieved sentence.

## What you hand in

One page, submitted to Canvas as this notebook (*File → Download → .ipynb*) after you have run every cell and answered every ✏️ prompt above.

1. Your Part 1 prediction and the two measured chunk counts.
2. Your Part 2 predictions (article-level and sentence-level, both queries) next to what actually came back.
3. The two Part 3 answers (without and with context), and one sentence on what the no-context answer got wrong.
4. One sentence: for a corpus like this one, would you chunk by article or by sentence, and why?

## My answers

**1. Part 1.** I predicted 4 article-level and 22 sentence-level chunks. Measured: 4 and 22.

**2. Part 2.**

| Query | Chunking | My prediction | Result |
|---|---|---|---|
| literal | article | cards-lost | cards-lost (0.869), correct |
| literal | sentence | cards-lost:5 "instant card for 3000 tenge" | cards-lost:4 "replacement card is issued free of charge" (0.851), needle was second (0.848) |
| paraphrase | article | cards-lost | cards-expiry (0.881) |
| paraphrase | sentence | may change | cards-expiry:4 "new one then costs 1500 tenge" (0.864), needle not in top 3 |

**3. Part 3.**

Without context: "I'm sorry, but I cannot provide you with that information as it may violate our terms of service to answer questions about specific products or services. However, you can usually find this information on the official website of the company or contact their customer support for assistance."

With context: "The cost of an instant replacement card at the branch is 3000 tenge."

The no-context answer did not give the price (3000 tenge) and made up a reason about terms of service, instead of just saying that it does not know this bank's rules.

**4. Chunking.** For this corpus I would chunk by article, because the articles are short (5-6 sentences) and fit into the prompt easily, and in my run article-level found the right article for the literal query while sentence-level put the wrong sentence first.

## AI-use declaration

I used Claude Code (an AI assistant from Anthropic that works in the terminal) to help me with this lab.

What I did myself: I decided to run the lab locally instead of Colab, read the four articles, wrote my predictions before running each part (I discussed the reasoning with the assistant before), ran the notebook, compared my predictions with the results and chose the final answers.

What the AI helped with:

- Setup: installing the packages and Jupyter, making the virtual environment (`.venv`), downloading the two models and writing `requirements.txt`.
- Fixing an error: in my global Python the Jupyter kernel crashed when importing `sentence_transformers` (a conflict between `pyarrow` and `torch` on Windows). We did not change the lab code, we ran it in a clean `.venv` where this error does not happen.
- Text: adding the prediction and result cells, the table, and fixing my English.

All numbers and model answers in this notebook are from my own local run, I did not edit them.